# Chapter 14: Affine Epipolar Geometry

Source orientation: printed pages 344-362; PDF pages 362-380.

This notebook is an original computational treatment of the chapter. The source pages were used to identify the chapter structure and terminology: affine cameras, parallel epipolar line families, the affine fundamental matrix, affine estimation as hyperplane fitting, triangulation by exact Sampson correction, affine reconstruction, bas-relief ambiguity, and the recoverable motion parameters. The prose, code, diagrams, and data below are original and use deterministic synthetic geometry rather than textbook figures, screenshots, page crops, or copied examples.

## Chapter Goal

The goal is to make the affine two-view case inspectable: when projection rays are parallel, the epipoles move to infinity, all epipolar lines in each image become parallel, and the fundamental matrix collapses to

$$
F_A = \begin{bmatrix}0&0&a\\0&0&b\\c&d&e\end{bmatrix}.
$$

That special form is not just fewer symbols. It changes the numerical problem: a point correspondence contributes the linear constraint

$$
a x' + b y' + c x + d y + e = 0,
$$

so estimation, optimal correction, and many diagnostics are ordinary linear algebra in the four measured image coordinates $(x',y',x,y)$. The chapter also has an important negative lesson: two affine views can recover parallax products, but not every combination of depth and out-of-plane turn that produced them.

## Computational Translation Guide

| Book idea | Computational object in this notebook | What to inspect |
| --- | --- | --- |
| Affine camera | A $3\times4$ camera whose last row is $(0,0,0,1)$ | Image coordinates are linear functions of world coordinates; projection rays are parallel. |
| Epipole at infinity | Null vector of $F_A$ or $F_A^T$ with last homogeneous coordinate zero | Epipolar lines have one shared orientation instead of a finite fan. |
| Affine fundamental matrix | Five-vector $(a,b,c,d,e)$ embedded in a rank-two $3\times3$ matrix | The zero pattern and the residual $x'^T F_A x$. |
| Gold-standard affine estimate | Orthogonal regression to a hyperplane in $(x',y',x,y)$ | The normal vector from SVD and the point-to-plane distances. |
| Triangulation correction | Orthogonal projection of a noisy correspondence onto that hyperplane | The corrected pair satisfies the epipolar equation exactly. |
| Relief/parallax | Difference between a plane-induced affine transfer and the observed second-view point | Small relief gives weak line direction information. |
| Bas-relief ambiguity | Constant product $\Delta Z\sin\rho$ | Different depth-turn pairs produce the same affine parallax. |

## Library Routing

| Concept | Representation | Library route | Why this route |
| --- | --- | --- | --- |
| Affine camera rays and parallel epipolar lines | Static two-view construction with measured points and line families | Matplotlib + NumPy | The concept is planar incidence and shared direction; a durable labeled PNG is clearer than a heavy 3D renderer. |
| Affine fundamental matrix identities | Exact zero pattern, residual identity, and null vectors | SymPy | The chapter claim is algebraic and can be checked exactly before using floating-point data. |
| Estimating $F_A$ | SVD of a correspondence matrix and an orthogonal hyperplane fit | NumPy linear algebra | The chapter's algorithmic simplification is precisely that the optimum is an SVD problem. |
| Parallax degeneracy and triangulation correction | Line/parallax diagrams and residual bars | Matplotlib + NumPy | The learner needs to compare measured and corrected image coordinates directly. |
| Depth-turn ambiguity | Interactive surface of $\Delta x = \Delta Z\sin\rho$ | Plotly HTML | A parameter surface makes the one-parameter ambiguity visible without requiring textbook images. |
| Artifact checks | JSON and file-size assertions | course `utils.artifacts` | Keeps generated paths book-local and validates reproducibility. |

## Route Through The Chapter

1. Build a synthetic affine stereo pair and verify the exact affine fundamental matrix.
2. Draw why the epipolar lines are parallel and how the line direction is encoded.
3. Estimate $F_A$ from noisy correspondences as a hyperplane in four measurement coordinates.
4. Use the minimal four-point story to show how relief creates parallax and why coplanarity is degenerate.
5. Correct a noisy correspondence by the exact affine Sampson projection, then recover affine coordinates from two images.
6. Explore the bas-relief ambiguity and the motion parameters that remain visible in $F_A$.

In [ ]:
from pathlib import Path
import json
import math
import sys

import matplotlib.pyplot as plt
import numpy as np
import plotly.graph_objects as go
import sympy as sp

BOOK_ROOT = Path.cwd()
for candidate in [BOOK_ROOT, *BOOK_ROOT.parents]:
    if (candidate / "00-book-index.ipynb").exists() and (candidate / "utils").exists():
        BOOK_ROOT = candidate
        break
else:
    raise RuntimeError("Could not find the MVG book root")

if str(BOOK_ROOT) not in sys.path:
    sys.path.insert(0, str(BOOK_ROOT))

from utils.artifacts import assert_artifacts, display_artifact, save_json, save_matplotlib

TOPIC = "chapter-14"
ARTIFACT_ROOT = BOOK_ROOT / "artifacts" / TOPIC
ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)

def save_plotly_html_chapter(fig, *parts):
    path = ARTIFACT_ROOT.joinpath(*parts)
    path.parent.mkdir(parents=True, exist_ok=True)
    fig.write_html(str(path), include_plotlyjs=True, full_html=True)
    return path

rng = np.random.default_rng(1401)
artifact_paths = []
check_data = {}

plt.rcParams.update({
    "figure.dpi": 120,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size": 10,
})

In [ ]:
def make_affine_camera(M, t):
    P = np.zeros((3, 4), dtype=float)
    P[:2, :3] = np.asarray(M, dtype=float)
    P[:2, 3] = np.asarray(t, dtype=float)
    P[2, 3] = 1.0
    return P


def affine_project(P, X):
    X = np.asarray(X, dtype=float)
    Xh = np.column_stack([X, np.ones(len(X))])
    xh = Xh @ P.T
    return xh[:, :2] / xh[:, 2:3]


def homogenize_2d(x):
    x = np.asarray(x, dtype=float)
    return np.column_stack([x, np.ones(len(x))])


def affine_F_from_canonical_camera(M, t):
    m11, m12, m13 = M[0]
    m21, m22, m23 = M[1]
    t1, t2 = t
    a = m23
    b = -m13
    c = m13 * m21 - m11 * m23
    d = m13 * m22 - m12 * m23
    e = m13 * t2 - m23 * t1
    return np.array([[0.0, 0.0, a], [0.0, 0.0, b], [c, d, e]], dtype=float)


def F_to_vector(F):
    return np.array([F[0, 2], F[1, 2], F[2, 0], F[2, 1], F[2, 2]], dtype=float)


def vector_to_F(f):
    a, b, c, d, e = np.asarray(f, dtype=float)
    return np.array([[0.0, 0.0, a], [0.0, 0.0, b], [c, d, e]], dtype=float)


def align_vector_scale(f, reference):
    f = np.asarray(f, dtype=float)
    reference = np.asarray(reference, dtype=float)
    return f * (np.dot(f, reference) / np.dot(f, f))


def affine_residuals(F, x1, x2):
    a, b, c, d, e = F_to_vector(F)
    return a * x2[:, 0] + b * x2[:, 1] + c * x1[:, 0] + d * x1[:, 1] + e


def estimate_affine_F_linear(x1, x2):
    A = np.column_stack([x2[:, 0], x2[:, 1], x1[:, 0], x1[:, 1], np.ones(len(x1))])
    _, singular_values, Vt = np.linalg.svd(A, full_matrices=False)
    return vector_to_F(Vt[-1]), singular_values


def estimate_affine_F_gold_standard(x1, x2):
    X = np.column_stack([x2[:, 0], x2[:, 1], x1[:, 0], x1[:, 1]])
    centroid = X.mean(axis=0)
    A = X - centroid
    _, singular_values, Vt = np.linalg.svd(A, full_matrices=False)
    N = Vt[-1]
    e = -float(N @ centroid)
    return vector_to_F(np.r_[N, e]), singular_values, centroid


def line_segment(line, xlim, ylim, samples=200):
    a, b, c = np.asarray(line, dtype=float)
    xs = np.linspace(xlim[0], xlim[1], samples)
    if abs(b) > 1e-12:
        ys = -(a * xs + c) / b
        mask = (ys >= ylim[0]) & (ys <= ylim[1])
        return xs[mask], ys[mask]
    ys = np.linspace(ylim[0], ylim[1], samples)
    xs = np.full_like(ys, -c / a)
    mask = (xs >= xlim[0]) & (xs <= xlim[1])
    return xs[mask], ys[mask]


def draw_line(ax, line, xlim, ylim, **kwargs):
    xs, ys = line_segment(line, xlim, ylim)
    if len(xs) > 1:
        ax.plot(xs, ys, **kwargs)


def set_equal_image_axes(ax, points, margin=0.35):
    pts = np.asarray(points)
    xlim = (float(pts[:, 0].min() - margin), float(pts[:, 0].max() + margin))
    ylim = (float(pts[:, 1].min() - margin), float(pts[:, 1].max() + margin))
    ax.set_xlim(*xlim)
    ax.set_ylim(*ylim)
    ax.set_aspect("equal", adjustable="box")
    ax.grid(True, alpha=0.22)
    return xlim, ylim


def affine_transform_from_3_points(src, dst):
    rows = []
    rhs = []
    for (x, y), (u, v) in zip(src, dst):
        rows.append([x, y, 1.0, 0.0, 0.0, 0.0])
        rows.append([0.0, 0.0, 0.0, x, y, 1.0])
        rhs.extend([u, v])
    params = np.linalg.solve(np.asarray(rows), np.asarray(rhs))
    return np.array([[params[0], params[1], params[2]], [params[3], params[4], params[5]], [0.0, 0.0, 1.0]])


def apply_affine_H(H, x):
    arr = np.asarray(x, dtype=float)
    if arr.ndim == 2:
        xh = homogenize_2d(arr)
        yh = xh @ H.T
        return yh[:, :2] / yh[:, 2:3]
    yh = H @ np.r_[arr, 1.0]
    return yh[:2] / yh[2]


def affine_sampson_correct(F, x1_measured, x2_measured):
    f = F_to_vector(F)
    N = f[:4]
    e = f[4]
    X = np.r_[x2_measured, x1_measured]
    residual = float(N @ X + e)
    correction = residual / float(N @ N) * N
    X_hat = X - correction
    return X_hat[2:], X_hat[:2], residual, correction


def reconstruct_affine_coordinates(x1, x2, basis_indices):
    b0, b1, b2, b3 = basis_indices
    E1 = np.column_stack([x1[b1] - x1[b0], x1[b2] - x1[b0], x1[b3] - x1[b0]])
    E2 = np.column_stack([x2[b1] - x2[b0], x2[b2] - x2[b0], x2[b3] - x2[b0]])
    A = np.vstack([E1, E2])
    coords = []
    for p1, p2 in zip(x1, x2):
        rhs = np.r_[p1 - x1[b0], p2 - x2[b0]]
        coord, *_ = np.linalg.lstsq(A, rhs, rcond=None)
        coords.append(coord)
    return np.asarray(coords)


def motion_parameters_from_F(F):
    a, b, c, d, _ = F_to_vector(F)
    phi = math.atan2(b, a)
    phi_minus_theta = math.atan2(d, c)
    theta = math.atan2(math.sin(phi - phi_minus_theta), math.cos(phi - phi_minus_theta))
    scale = math.sqrt((c * c + d * d) / (a * a + b * b))
    return {"phi_rad": phi, "phi_minus_theta_rad": phi_minus_theta, "theta_rad": theta, "scale": scale}

In [ ]:
# A deterministic affine stereo pair in canonical form.
M2 = np.array([[1.08, -0.16, 0.55], [0.12, 0.92, -0.35]])
t2 = np.array([0.35, -0.18])
P1 = make_affine_camera(np.array([[1.0, 0.0, 0.0], [0.0, 1.0, 0.0]]), np.array([0.0, 0.0]))
P2 = make_affine_camera(M2, t2)
F_true = affine_F_from_canonical_camera(M2, t2)
f_true = F_to_vector(F_true)

scene_points = np.array([
    [-0.85, -0.65, -0.25], [-0.25, -0.70, 0.45], [0.35, -0.62, -0.10], [0.85, -0.48, 0.72],
    [-0.78, -0.05, 0.35], [-0.18, -0.04, -0.45], [0.40, 0.05, 0.18], [0.88, 0.08, -0.60],
    [-0.70, 0.58, -0.15], [-0.12, 0.62, 0.78], [0.50, 0.57, -0.35], [0.92, 0.68, 0.30],
])
x1 = affine_project(P1, scene_points)
x2 = affine_project(P2, scene_points)

noise_sigma = 0.015
x1_noisy = x1 + rng.normal(scale=noise_sigma, size=x1.shape)
x2_noisy = x2 + rng.normal(scale=noise_sigma, size=x2.shape)

residuals_true = affine_residuals(F_true, x1, x2)
F_linear, linear_singular_values = estimate_affine_F_linear(x1_noisy, x2_noisy)
F_gold, gold_singular_values, gold_centroid = estimate_affine_F_gold_standard(x1_noisy, x2_noisy)
f_linear_aligned = align_vector_scale(F_to_vector(F_linear), f_true)
f_gold_aligned = align_vector_scale(F_to_vector(F_gold), f_true)
F_gold_aligned = vector_to_F(f_gold_aligned)

check_data.update({
    "noise_free_max_affine_epipolar_residual": float(np.max(np.abs(residuals_true))),
    "linear_estimate_vector_error": float(np.linalg.norm(f_linear_aligned - f_true)),
    "gold_standard_vector_error": float(np.linalg.norm(f_gold_aligned - f_true)),
    "affine_F_rank": int(np.linalg.matrix_rank(F_true, tol=1e-12)),
})

F_true

## 1. Affine Cameras Make Epipolar Lines Parallel

A finite perspective camera sends a 3D point to an image along rays through a finite center. An affine camera sends all points along one common projection direction. In a two-view affine pair, the back-projected rays from the first image are parallel 3D lines; their images in the second view are therefore parallel image lines. Algebraically this is the statement that $F_A x$ has first two coordinates $(a,b)$ independent of the point $x=(x,y,1)^T$.

In the figure below, inspect the two image panels: every epipolar line in a given image shares its normal vector. The epipole is not a plotted finite point because it is an ideal point on the line at infinity.

In [ ]:
fig = plt.figure(figsize=(13.0, 4.2))
grid = fig.add_gridspec(1, 3, width_ratios=[1.05, 1.0, 1.0])

ax0 = fig.add_subplot(grid[0])
for X in scene_points[::2]:
    ax0.scatter(X[0], X[2], color="#2f6f9f", s=28)
    ax0.arrow(X[0], X[2], 0.0, -0.72, width=0.006, head_width=0.055, head_length=0.07, color="#6b7280", alpha=0.75, length_includes_head=True)
ax0.axhline(-1.05, color="#111827", linewidth=1.2)
ax0.text(-1.05, -1.18, "image plane for view 1", fontsize=9)
ax0.text(-1.05, 0.87, "parallel projection rays", fontsize=10, weight="bold")
ax0.set_xlabel("world X")
ax0.set_ylabel("world Z")
ax0.set_title("Affine camera: center at infinity")
ax0.set_aspect("equal", adjustable="box")
ax0.grid(True, alpha=0.22)

ax1 = fig.add_subplot(grid[1])
ax1.scatter(x1[:, 0], x1[:, 1], s=32, color="#1f77b4", label="view 1 points")
xlim1, ylim1 = set_equal_image_axes(ax1, x1)
for p2 in x2[::2]:
    line = F_true.T @ np.r_[p2, 1.0]
    draw_line(ax1, line, xlim1, ylim1, color="#d55e00", alpha=0.62, linewidth=1.0)
normal1 = np.array([F_true[2, 0], F_true[2, 1]])
normal1 = normal1 / np.linalg.norm(normal1)
origin1 = np.array([xlim1[0] + 0.18, ylim1[1] - 0.18])
ax1.arrow(origin1[0], origin1[1], 0.28 * normal1[0], 0.28 * normal1[1], color="#111827", width=0.008, head_width=0.055, length_includes_head=True)
ax1.text(origin1[0] + 0.03, origin1[1] - 0.10, "shared normal (c,d)", fontsize=8)
ax1.set_title("Lines in image 1: $F_A^T x'$ ")
ax1.set_xlabel("x")
ax1.set_ylabel("y")

ax2p = fig.add_subplot(grid[2])
ax2p.scatter(x2[:, 0], x2[:, 1], s=32, color="#009e73", label="view 2 points")
xlim2, ylim2 = set_equal_image_axes(ax2p, x2)
for p1 in x1[::2]:
    line = F_true @ np.r_[p1, 1.0]
    draw_line(ax2p, line, xlim2, ylim2, color="#cc79a7", alpha=0.62, linewidth=1.0)
normal2 = np.array([F_true[0, 2], F_true[1, 2]])
normal2 = normal2 / np.linalg.norm(normal2)
origin2 = np.array([xlim2[0] + 0.18, ylim2[1] - 0.18])
ax2p.arrow(origin2[0], origin2[1], 0.28 * normal2[0], 0.28 * normal2[1], color="#111827", width=0.008, head_width=0.055, length_includes_head=True)
ax2p.text(origin2[0] + 0.03, origin2[1] - 0.10, "shared normal (a,b)", fontsize=8)
ax2p.set_title("Lines in image 2: $F_A x$")
ax2p.set_xlabel("x'")
ax2p.set_ylabel("y'")

fig.suptitle("Affine epipolar geometry replaces a finite epipolar fan by two parallel line families", y=1.02, fontsize=13)
fig.tight_layout()
parallel_lines_path = save_matplotlib(fig, TOPIC, "figures", "affine-cameras-parallel-epipolar-lines.png", dpi=170)
plt.close(fig)
artifact_paths.append(parallel_lines_path)
display_artifact(parallel_lines_path, width=980)

line_normals_second = np.array([(F_true @ np.r_[p, 1.0])[:2] for p in x1])
line_normals_second /= np.linalg.norm(line_normals_second, axis=1, keepdims=True)
check_data["parallel_line_normal_spread_second_view"] = float(np.max(np.linalg.norm(line_normals_second - line_normals_second[0], axis=1)))

## 2. The Affine Fundamental Matrix Has an Exact Zero Pattern

For the canonical pair

$$
P = \begin{bmatrix}1&0&0&0\\0&1&0&0\\0&0&0&1\end{bmatrix},\qquad
P' = \begin{bmatrix}m_{11}&m_{12}&m_{13}&t_1\\m_{21}&m_{22}&m_{23}&t_2\\0&0&0&1\end{bmatrix},
$$

the affine fundamental matrix can be checked directly. The symbolic cell below verifies three claims used throughout the notebook: the residual vanishes for every 3D point, the right epipole has the form $(-d,c,0)^T$, and the left epipole has the form $(-b,a,0)^T$.

In [ ]:
m11, m12, m13, m21, m22, m23, t1, t2 = sp.symbols("m11 m12 m13 m21 m22 m23 t1 t2")
X, Y, Z = sp.symbols("X Y Z")
a = m23
b = -m13
c = m13 * m21 - m11 * m23
d = m13 * m22 - m12 * m23
e = m13 * t2 - m23 * t1
F_sym = sp.Matrix([[0, 0, a], [0, 0, b], [c, d, e]])
x_sym = sp.Matrix([X, Y, 1])
xp_sym = sp.Matrix([m11 * X + m12 * Y + m13 * Z + t1, m21 * X + m22 * Y + m23 * Z + t2, 1])
residual_identity = sp.expand((xp_sym.T * F_sym * x_sym)[0])
right_epipole = sp.Matrix([-d, c, 0])
left_epipole = sp.Matrix([-b, a, 0])
right_null = sp.simplify(F_sym * right_epipole)
left_null = sp.simplify(F_sym.T * left_epipole)

assert residual_identity == 0
assert list(right_null) == [0, 0, 0]
assert list(left_null) == [0, 0, 0]

symbolic_summary = {
    "residual_identity": str(residual_identity),
    "right_epipole": [str(v) for v in right_epipole],
    "left_epipole": [str(v) for v in left_epipole],
    "zero_pattern": [[int(v == 0) for v in row] for row in F_sym.tolist()],
}
check_data["symbolic_affine_F_identity"] = symbolic_summary["residual_identity"]
symbolic_summary

## 3. Estimating $F_A$ Is Hyperplane Fitting

Each correspondence becomes a point $X_i=(x'_i,y'_i,x_i,y_i)$ in four-dimensional measurement space. The affine epipolar equation says these points lie on a hyperplane with normal $(a,b,c,d)$ and offset $e$. The linear nullspace solve fits the homogeneous five-vector. The affine Gold Standard solve centers the four-dimensional data and fits the least-squares orthogonal hyperplane by SVD.

The diagnostic below uses noisy correspondences. The true matrix is only used for validation; the estimator sees image coordinates.

In [ ]:
F_gold_scaled = vector_to_F(f_gold_aligned)
residuals_noisy_true = affine_residuals(F_true, x1_noisy, x2_noisy)
residuals_noisy_gold = affine_residuals(F_gold_scaled, x1_noisy, x2_noisy)

motion = motion_parameters_from_F(F_true)
axis_1 = np.array([F_true[2, 0], F_true[2, 1]])
axis_2 = np.array([F_true[0, 2], F_true[1, 2]])
axis_1 /= np.linalg.norm(axis_1)
axis_2 /= np.linalg.norm(axis_2)

fig, axes = plt.subplots(1, 3, figsize=(13.2, 3.9))

im = axes[0].imshow(F_true, cmap="coolwarm", vmin=-np.max(np.abs(F_true)), vmax=np.max(np.abs(F_true)))
for i in range(3):
    for j in range(3):
        axes[0].text(j, i, f"{F_true[i, j]:.2f}", ha="center", va="center", color="black", fontsize=9)
axes[0].set_xticks(range(3), ["x", "y", "1"])
axes[0].set_yticks(range(3), ["x'", "y'", "1"])
axes[0].set_title("$F_A$ zero pattern and entries")
fig.colorbar(im, ax=axes[0], fraction=0.046, pad=0.04)

sv_labels = np.arange(1, len(gold_singular_values) + 1)
axes[1].semilogy(sv_labels, gold_singular_values, marker="o", color="#2f6f9f", label="centered 4D data")
axes[1].semilogy(np.arange(1, len(linear_singular_values) + 1), linear_singular_values, marker="s", color="#d55e00", alpha=0.75, label="homogeneous design")
axes[1].set_xlabel("singular value index")
axes[1].set_ylabel("singular value")
axes[1].set_title("SVD exposes the fitted hyperplane")
axes[1].grid(True, alpha=0.25)
axes[1].legend(fontsize=8)

bins = np.linspace(min(residuals_noisy_true.min(), residuals_noisy_gold.min()), max(residuals_noisy_true.max(), residuals_noisy_gold.max()), 10)
axes[2].hist(residuals_noisy_true, bins=bins, alpha=0.6, color="#6b7280", label="true $F_A$ on noisy data")
axes[2].hist(residuals_noisy_gold, bins=bins, alpha=0.65, color="#009e73", label="fitted $F_A$")
axes[2].axvline(0.0, color="#111827", linewidth=1.0)
axes[2].set_xlabel("affine epipolar residual")
axes[2].set_ylabel("count")
axes[2].set_title("Residuals are point-to-plane distances up to scale")
axes[2].legend(fontsize=8)

fig.suptitle("Affine fundamental matrix estimation is a four-coordinate hyperplane fit", y=1.03, fontsize=13)
fig.tight_layout()
estimation_path = save_matplotlib(fig, TOPIC, "figures", "affine-fundamental-matrix-hyperplane-fit.png", dpi=170)
plt.close(fig)
artifact_paths.append(estimation_path)
display_artifact(estimation_path, width=980)

check_data.update({
    "gold_standard_mean_abs_residual_on_noisy_points": float(np.mean(np.abs(residuals_noisy_gold))),
    "gold_standard_smallest_singular_value": float(gold_singular_values[-1]),
    "motion_phi_degrees": float(np.degrees(motion["phi_rad"])),
    "motion_theta_degrees": float(np.degrees(motion["theta_rad"])),
    "motion_scale_from_F": float(motion["scale"]),
})

## 4. Epipolar-Line Direction and Recoverable Motion

The affine matrix stores two line normals: $(a,b)$ for lines in the second image and $(c,d)$ for lines in the first image. In the weak-perspective motion interpretation, their angles determine the projected out-of-plane rotation axis in each image; the difference between those angles gives the in-plane cyclo-rotation. The magnitude of the out-of-plane turn is not determined by two affine views.

The next diagram draws the same line families as directions rather than as constraints. The arrows are normal to the epipolar lines, so they are parallel to the projected rotation axis in each image.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10.6, 4.2))

for ax, pts, partner_pts, title, normal, F_side in [
    (axes[0], x1, x2, "image 1: normal $(c,d)$", axis_1, "left"),
    (axes[1], x2, x1, "image 2: normal $(a,b)$", axis_2, "right"),
]:
    ax.scatter(pts[:, 0], pts[:, 1], s=35, color="#2f6f9f" if F_side == "left" else "#009e73")
    xlim, ylim = set_equal_image_axes(ax, pts, margin=0.45)
    for p in partner_pts[::3]:
        line = F_true.T @ np.r_[p, 1.0] if F_side == "left" else F_true @ np.r_[p, 1.0]
        draw_line(ax, line, xlim, ylim, color="#d55e00" if F_side == "left" else "#cc79a7", alpha=0.68, linewidth=1.2)
    center = np.array([(xlim[0] + xlim[1]) / 2, (ylim[0] + ylim[1]) / 2])
    ax.arrow(center[0], center[1], 0.48 * normal[0], 0.48 * normal[1], color="#111827", width=0.01, head_width=0.07, length_includes_head=True)
    ax.text(center[0] + 0.04, center[1] + 0.06, "axis projection", fontsize=9, weight="bold")
    ax.set_title(title)
    ax.set_xlabel("x" if F_side == "left" else "x'")
    ax.set_ylabel("y" if F_side == "left" else "y'")

phi_deg = np.degrees(motion["phi_rad"])
phi_minus_theta_deg = np.degrees(motion["phi_minus_theta_rad"])
theta_deg = np.degrees(motion["theta_rad"])
fig.suptitle(f"Motion visible in $F_A$: phi={phi_deg:.1f} deg, phi-theta={phi_minus_theta_deg:.1f} deg, theta={theta_deg:.1f} deg; rho remains hidden", y=1.02, fontsize=12)
fig.tight_layout()
motion_path = save_matplotlib(fig, TOPIC, "figures", "epipolar-line-direction-motion-parameters.png", dpi=170)
plt.close(fig)
artifact_paths.append(motion_path)
display_artifact(motion_path, width=900)

## 5. Minimal Four-Point Parallax and Degeneracy

With three points on a scene plane, the two images determine a plane-induced affine transfer $H_A$. A fourth point off the plane reveals itself by its parallax vector: the difference between its actual second-view measurement and the point predicted by $H_A$. That parallax determines the epipolar line direction. If the fourth point moves into the plane, the parallax vector goes to zero and the minimal configuration loses the information needed to determine $F_A$.

The figure varies only the fourth point's relief. The left panel shows the actual point separating from its plane-transfer prediction; the right panel shows the linear growth of parallax with relief.

In [ ]:
plane_points = np.array([[0.0, 0.0, 0.0], [1.0, 0.05, 0.0], [0.10, 0.95, 0.0]])
relief_values = np.linspace(0.0, 1.25, 8)
base_fourth_xy = np.array([0.58, 0.42])
all_plane_x1 = affine_project(P1, plane_points)
all_plane_x2 = affine_project(P2, plane_points)
H_plane = affine_transform_from_3_points(all_plane_x1, all_plane_x2)

predicted = []
observed = []
parallax_norms = []
for relief in relief_values:
    X4 = np.array([[base_fourth_xy[0], base_fourth_xy[1], relief]])
    x1_four = affine_project(P1, X4)[0]
    x2_four = affine_project(P2, X4)[0]
    x2_plane = apply_affine_H(H_plane, x1_four)
    predicted.append(x2_plane)
    observed.append(x2_four)
    parallax_norms.append(float(np.linalg.norm(x2_four - x2_plane)))
predicted = np.asarray(predicted)
observed = np.asarray(observed)
parallax_norms = np.asarray(parallax_norms)
expected_slope = float(np.linalg.norm(M2[:, 2]))

fig, axes = plt.subplots(1, 2, figsize=(11.2, 4.2))
axes[0].scatter(all_plane_x2[:, 0], all_plane_x2[:, 1], color="#6b7280", s=42, label="three plane points")
axes[0].scatter(predicted[:, 0], predicted[:, 1], color="#2f6f9f", s=28, label="$H_A x_4$")
axes[0].scatter(observed[:, 0], observed[:, 1], color="#d55e00", s=28, label="observed $x'_4$")
for i, relief in enumerate(relief_values):
    axes[0].arrow(predicted[i, 0], predicted[i, 1], observed[i, 0] - predicted[i, 0], observed[i, 1] - predicted[i, 1], color="#111827", alpha=0.65, head_width=0.025, length_includes_head=True)
    if i in {0, len(relief_values) - 1}:
        axes[0].text(observed[i, 0] + 0.02, observed[i, 1] + 0.02, f"Z={relief:.2f}", fontsize=8)
set_equal_image_axes(axes[0], np.vstack([all_plane_x2, predicted, observed]), margin=0.28)
axes[0].set_title("Parallax relative to the plane transfer")
axes[0].set_xlabel("x'")
axes[0].set_ylabel("y'")
axes[0].legend(fontsize=8)

axes[1].plot(relief_values, parallax_norms, marker="o", color="#111827", label="measured parallax norm")
axes[1].plot(relief_values, expected_slope * relief_values, linestyle="--", color="#009e73", label="$||M_{:,3}|| Z$")
axes[1].axhline(noise_sigma, color="#d55e00", linestyle=":", label="one noise sigma")
axes[1].set_xlabel("relief of fourth point")
axes[1].set_ylabel("parallax length")
axes[1].set_title("Coplanarity erases the line direction")
axes[1].grid(True, alpha=0.25)
axes[1].legend(fontsize=8)

fig.suptitle("The minimal affine epipolar configuration needs off-plane parallax", y=1.02, fontsize=13)
fig.tight_layout()
parallax_path = save_matplotlib(fig, TOPIC, "figures", "minimal-parallax-relief-conditioning.png", dpi=170)
plt.close(fig)
artifact_paths.append(parallax_path)
display_artifact(parallax_path, width=940)

check_data.update({
    "minimal_parallax_at_zero_relief": float(parallax_norms[0]),
    "minimal_parallax_slope_error": float(np.max(np.abs(parallax_norms - expected_slope * relief_values))),
})

## 6. Affine Triangulation as Exact Sampson Correction

For a general fundamental matrix, the Sampson correction is a first-order approximation to geometric correction. In the affine case the constraint is already a hyperplane in $(x',y',x,y)$, so the closest corrected correspondence is the orthogonal projection onto that hyperplane. Once the pair is corrected, any affine triangulation or affine-coordinate reconstruction can use the corrected image measurements.

The next cell perturbs one correspondence, projects the four measured coordinates onto the affine epipolar hyperplane, and displays the correction in both images.

In [ ]:
sample_index = 9
x1_measured = x1[sample_index] + np.array([0.035, -0.022])
x2_measured = x2[sample_index] + np.array([-0.018, 0.030])
x1_corrected, x2_corrected, residual_before, correction4 = affine_sampson_correct(F_true, x1_measured, x2_measured)
residual_after = float(affine_residuals(F_true, x1_corrected[None, :], x2_corrected[None, :])[0])
N_true = f_true[:4]
expected_correction_norm_sq = residual_before**2 / float(N_true @ N_true)
actual_correction_norm_sq = float(correction4 @ correction4)

fig, axes = plt.subplots(1, 3, figsize=(13.0, 4.1), width_ratios=[1.0, 1.0, 0.85])

axes[0].scatter(x1[:, 0], x1[:, 1], s=26, color="#cbd5e1", label="other exact points")
axes[0].scatter([x1_measured[0]], [x1_measured[1]], color="#d55e00", s=55, label="measured")
axes[0].scatter([x1_corrected[0]], [x1_corrected[1]], color="#009e73", s=55, label="corrected")
axes[0].arrow(x1_measured[0], x1_measured[1], x1_corrected[0] - x1_measured[0], x1_corrected[1] - x1_measured[1], color="#111827", head_width=0.018, length_includes_head=True)
xlim, ylim = set_equal_image_axes(axes[0], np.vstack([x1, x1_measured, x1_corrected]), margin=0.25)
draw_line(axes[0], F_true.T @ np.r_[x2_corrected, 1.0], xlim, ylim, color="#2f6f9f", linewidth=1.4)
axes[0].set_title("image 1 correction")
axes[0].set_xlabel("x")
axes[0].set_ylabel("y")
axes[0].legend(fontsize=8)

axes[1].scatter(x2[:, 0], x2[:, 1], s=26, color="#cbd5e1")
axes[1].scatter([x2_measured[0]], [x2_measured[1]], color="#d55e00", s=55, label="measured")
axes[1].scatter([x2_corrected[0]], [x2_corrected[1]], color="#009e73", s=55, label="corrected")
axes[1].arrow(x2_measured[0], x2_measured[1], x2_corrected[0] - x2_measured[0], x2_corrected[1] - x2_measured[1], color="#111827", head_width=0.018, length_includes_head=True)
xlim, ylim = set_equal_image_axes(axes[1], np.vstack([x2, x2_measured, x2_corrected]), margin=0.25)
draw_line(axes[1], F_true @ np.r_[x1_corrected, 1.0], xlim, ylim, color="#cc79a7", linewidth=1.4)
axes[1].set_title("image 2 correction")
axes[1].set_xlabel("x'")
axes[1].set_ylabel("y'")
axes[1].legend(fontsize=8)

axes[2].bar(["before", "after"], [abs(residual_before), abs(residual_after)], color=["#d55e00", "#009e73"])
axes[2].set_yscale("log")
axes[2].set_ylabel("absolute affine residual")
axes[2].set_title("projection onto the hyperplane")
axes[2].grid(True, axis="y", alpha=0.25)

fig.suptitle("Affine triangulation correction is exact orthogonal projection in measurement space", y=1.02, fontsize=13)
fig.tight_layout()
triangulation_path = save_matplotlib(fig, TOPIC, "figures", "affine-sampson-triangulation-correction.png", dpi=170)
plt.close(fig)
artifact_paths.append(triangulation_path)
display_artifact(triangulation_path, width=980)

check_data.update({
    "sampson_residual_before": float(residual_before),
    "sampson_residual_after": float(residual_after),
    "sampson_correction_norm_sq": actual_correction_norm_sq,
    "sampson_expected_norm_sq": float(expected_correction_norm_sq),
})

## 7. Affine Reconstruction from a Basis

Four non-coplanar scene points define an affine coordinate frame. Their two image projections supply enough linear equations to recover the affine coordinates of additional points. This is not a metric reconstruction: lengths and angles are not Euclidean truths. It is an affine reconstruction, so ratios along parallel directions and linear coordinate relations are meaningful.

The cell below uses a basis point as origin and three basis vectors. Every other point is reconstructed by solving the two-view affine-coordinate equations.

In [ ]:
basis_points = np.array([
    [0.0, 0.0, 0.0],
    [1.0, 0.0, 0.0],
    [0.0, 1.0, 0.0],
    [0.0, 0.0, 1.0],
    [0.35, 0.22, 0.62],
    [0.72, 0.18, 0.30],
    [0.16, 0.66, 0.54],
    [0.52, 0.72, 0.20],
    [0.42, 0.40, 0.82],
])
xb1 = affine_project(P1, basis_points)
xb2 = affine_project(P2, basis_points)
recovered_coords = reconstruct_affine_coordinates(xb1, xb2, [0, 1, 2, 3])
reconstruction_rmse = float(np.sqrt(np.mean(np.sum((recovered_coords - basis_points) ** 2, axis=1))))

fig = plt.figure(figsize=(10.6, 4.6))
ax = fig.add_subplot(1, 2, 1, projection="3d")
ax.scatter(basis_points[:, 0], basis_points[:, 1], basis_points[:, 2], color="#2f6f9f", s=40, label="true affine coords")
ax.scatter(recovered_coords[:, 0], recovered_coords[:, 1], recovered_coords[:, 2], color="#d55e00", marker="x", s=46, label="recovered")
for i in range(1, 4):
    ax.plot([0, basis_points[i, 0]], [0, basis_points[i, 1]], [0, basis_points[i, 2]], color="#111827", linewidth=1.5)
ax.set_xlabel("X")
ax.set_ylabel("Y")
ax.set_zlabel("Z")
ax.set_title("Recovered affine coordinates")
ax.legend(fontsize=8)

ax2 = fig.add_subplot(1, 2, 2)
errors = np.linalg.norm(recovered_coords - basis_points, axis=1)
ax2.bar(np.arange(len(errors)), errors, color="#009e73")
ax2.set_xlabel("point index")
ax2.set_ylabel("coordinate error")
ax2.set_title("Two affine views recover the basis coordinates")
ax2.set_yscale("log")
ax2.grid(True, axis="y", alpha=0.25)

fig.suptitle("Affine reconstruction is linear once a non-coplanar basis is fixed", y=1.02, fontsize=13)
fig.tight_layout()
reconstruction_path = save_matplotlib(fig, TOPIC, "figures", "affine-reconstruction-basis-coordinates.png", dpi=170)
plt.close(fig)
artifact_paths.append(reconstruction_path)
display_artifact(reconstruction_path, width=900)

check_data["affine_basis_reconstruction_rmse"] = reconstruction_rmse

## 8. Depth-Turn Bas-Relief Ambiguity

Under parallel projection, a point's image displacement caused by a depth difference and an out-of-plane turn depends on a product such as $\Delta Z\sin\rho$. A shallow object with a larger turn can match the same affine parallax as a deeper object with a smaller turn. This is why two affine views can stabilize a parallax product but cannot by themselves separate depth from the turn angle.

The interactive surface below plots $\Delta x = \Delta Z\sin\rho$. Move around the surface and inspect the constant-height curve: every point on that curve has the same affine parallax.

In [ ]:
rho_degrees = np.linspace(2.0, 70.0, 90)
depth_values = np.linspace(0.15, 3.0, 90)
RHO, DEPTH = np.meshgrid(np.radians(rho_degrees), depth_values)
PARALLAX = DEPTH * np.sin(RHO)
constant_parallax = 0.55
rho_curve = np.radians(np.linspace(12.0, 65.0, 120))
depth_curve = constant_parallax / np.sin(rho_curve)
valid = depth_curve <= depth_values.max()
rho_pair = np.radians(np.array([24.0, 48.0]))
depth_pair = constant_parallax / np.sin(rho_pair)
parallax_pair = depth_pair * np.sin(rho_pair)

fig_plotly = go.Figure()
fig_plotly.add_trace(go.Surface(
    x=rho_degrees,
    y=depth_values,
    z=PARALLAX,
    colorscale="Viridis",
    opacity=0.86,
    colorbar={"title": "parallax"},
    name="Delta Z sin rho",
))
fig_plotly.add_trace(go.Scatter3d(
    x=np.degrees(rho_curve[valid]),
    y=depth_curve[valid],
    z=np.full(np.count_nonzero(valid), constant_parallax),
    mode="lines",
    line={"color": "black", "width": 6},
    name="constant parallax family",
))
fig_plotly.add_trace(go.Scatter3d(
    x=np.degrees(rho_pair),
    y=depth_pair,
    z=parallax_pair,
    mode="markers+text",
    marker={"size": 5, "color": "red"},
    text=["deep/small turn", "shallow/large turn"],
    textposition="top center",
    name="same observed parallax",
))
fig_plotly.update_layout(
    title="Bas-relief family: depth and out-of-plane turn are confounded in affine stereo",
    scene={
        "xaxis_title": "turn angle rho (degrees)",
        "yaxis_title": "depth difference Delta Z",
        "zaxis_title": "image parallax",
    },
    margin={"l": 0, "r": 0, "t": 50, "b": 0},
    height=620,
)

bas_relief_path = save_plotly_html_chapter(fig_plotly, "interactive", "depth-turn-bas-relief-family.html")
artifact_paths.append(bas_relief_path)
display_artifact(bas_relief_path, width=920, height=640)

check_data.update({
    "bas_relief_constant_parallax_target": float(constant_parallax),
    "bas_relief_pair_parallax_difference": float(abs(parallax_pair[0] - parallax_pair[1])),
})

## Applied Lab: Stress the Relief Before Trusting the Estimate

A useful practice exercise is to keep the two affine cameras fixed, scale the scene's relief toward zero, add a small amount of image noise, and estimate $F_A$ repeatedly. The expected behavior is not mysterious: when relief is large, parallax gives a stable epipolar-line direction; when relief is tiny, the line direction is dominated by noise because the point set is almost planar.

The notebook's checks already include the zero-relief parallax value and the SVD residuals. To extend the lab, add a Monte Carlo loop over `relief_scale` and plot the angular error of `(a,b)` or `(c,d)`. The same machinery can be used with real tracked features, but synthetic data is the right first test because the true affine camera is known.

## Final Sanity Checks

The final cell gathers the invariant checks that matter for this chapter: matrix form and rank, epipolar residuals, parallel line direction, symbolic identities, exact affine Sampson correction, relief/parallax behavior, affine-coordinate reconstruction, Plotly artifact creation, and nonzero artifact sizes.

In [ ]:
invariant_summary = {
    "source_span": "printed pages 344-362; PDF pages 362-380",
    "artifacts": [str(path.relative_to(BOOK_ROOT)).replace("//", "/") for path in artifact_paths],
    "checks": check_data,
    "affine_F_vector": [float(v) for v in f_true],
    "motion_from_F": {key: float(value) for key, value in motion.items()},
}
summary_path = save_json(invariant_summary, TOPIC, "checks", "affine-epipolar-invariants.json")
artifact_paths.append(summary_path)
display_artifact(summary_path)

assert F_true.shape == (3, 3)
assert np.allclose(F_true[:2, :2], 0.0)
assert np.linalg.matrix_rank(F_true, tol=1e-12) == 2
assert check_data["noise_free_max_affine_epipolar_residual"] < 1e-12
assert check_data["parallel_line_normal_spread_second_view"] < 1e-12
assert check_data["symbolic_affine_F_identity"] == "0"
assert abs(check_data["sampson_residual_after"]) < 1e-12
assert abs(check_data["sampson_correction_norm_sq"] - check_data["sampson_expected_norm_sq"]) < 1e-14
assert check_data["minimal_parallax_at_zero_relief"] < 1e-12
assert check_data["minimal_parallax_slope_error"] < 1e-12
assert check_data["affine_basis_reconstruction_rmse"] < 1e-12
assert check_data["bas_relief_pair_parallax_difference"] < 1e-12
assert len(artifact_paths) >= 7
assert_artifacts(artifact_paths, min_bytes=1500)

final_sanity = {
    "artifact_count": len(artifact_paths),
    "max_noise_free_epipolar_residual": check_data["noise_free_max_affine_epipolar_residual"],
    "sampson_residual_after": check_data["sampson_residual_after"],
    "affine_reconstruction_rmse": check_data["affine_basis_reconstruction_rmse"],
    "bas_relief_pair_parallax_difference": check_data["bas_relief_pair_parallax_difference"],
}
final_sanity

## Takeaways

- Affine cameras push epipoles to infinity, so each image contains a parallel family of epipolar lines.
- The affine fundamental matrix has five nonzero degrees before scale, a built-in rank-two structure, and a residual that is linear in the four measured image coordinates.
- Estimating $F_A$ from noisy correspondences is orthogonal hyperplane fitting; the affine Sampson correction is exact for the same reason.
- Minimal four-point estimation depends on off-plane parallax. As relief tends to zero, the epipolar-line direction becomes poorly conditioned.
- Two affine views can recover affine structure once a basis is fixed, but they cannot separate depth from out-of-plane turn in the bas-relief family.